# Model the Air Quality Index with a gaussian process.

In [1]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from MuyGPyS._test.sampler import print_results
from MuyGPyS.neighbors import NN_Wrapper
from MuyGPyS.gp import MuyGPS
from MuyGPyS.gp.deformation import Isotropy, l2
from MuyGPyS.gp.hyperparameter import AnalyticScale, Parameter
from MuyGPyS.gp.kernels import Matern
from MuyGPyS.gp.noise import HomoscedasticNoise
from MuyGPyS.optimize import Bayes_optimize, L_BFGS_B_optimize
from MuyGPyS.optimize.batch import sample_batch
from MuyGPyS.optimize.loss import lool_fn

# Import and setup data

In [2]:
BASE_DIR = os.path.abspath("..") + "\\"
aqi_data = np.genfromtxt(BASE_DIR + "data\\biggest_day_aqi_coords_scaled.csv", delimiter=',', skip_header=1)
#print(aqi_data[:10])

In [3]:
#randomly split the data into features and responces, and those in turn into training and testing paritions.
aqi_x, aqi_y = aqi_data[:, :-1], aqi_data[:, -1]
aqi_x_train, aqi_x_test, aqi_y_train, aqi_y_test = train_test_split(aqi_x, aqi_y, test_size=0.1, random_state=24)

#use the mean of the training points to provide basic de-meaning of the points.
aqi_y_mean = aqi_y_train.mean()
aqi_y_train = aqi_y_train - aqi_y_mean

# Nearest Neighbor and Batches

In [4]:
#setup nearest nieghbors
nn_count = 30
nbrs_lookup = NN_Wrapper(aqi_x_train, nn_count, nn_method="exact", algorithm="ball_tree")


In [5]:
#for the sake of making use of batches, will use a batch value of 1500. this will include all of the data points.
batch_count = 1500
#get the indices of the training points and the indices of each points neighbors.
batch_indices, batch_nn_indices = sample_batch(
    nbrs_lookup, batch_count, len(aqi_x_train)
)

# setting and optimizing hyperparamters

In [6]:
#create our unoptimized muyGP object. set the ranges we want the hyperparameters to be optimized between.
aqi_muygps = MuyGPS(
    kernel=Matern(
        smoothness=Parameter("log_sample", (0.1, 4.0)),
        deformation=Isotropy(
            l2,
            length_scale=Parameter("log_sample", (0.01, 2.0)),
        ),
    ),
    noise=HomoscedasticNoise("sample", (1e-6, 1.0)),
    scale=AnalyticScale(),
)

In [7]:
#use the indices of the training points, and of the training points neighbors, together with the points' coordinates and values
#to calculate the crosswise distances between points and the pairwise distances between the neighbors
#and also the actual value of each point, and each point's neighbor.
(
    batch_crosswise_dists,
    batch_pairwise_dists,
    batch_ys,
    batch_nn_ys,
) = aqi_muygps.make_train_tensors(
    batch_indices,
    batch_nn_indices,
    aqi_x_train,
    aqi_y_train,
)

In [8]:
#optimize the parameters for 15 random initilization runs, and then 25 optimizing runs.
aqi_muygps_optimized = Bayes_optimize(
    aqi_muygps,
    batch_ys,
    batch_nn_ys,
    batch_crosswise_dists,
    batch_pairwise_dists,
    loss_fn=lool_fn,
    verbose=True,
    random_state=24,
    init_points=15,
    n_iter=25,
)

# aqi_muygps_optimized = L_BFGS_B_optimize(
#     aqi_muygps,
#     batch_ys,
#     batch_nn_ys,
#     batch_crosswise_dists,
#     batch_pairwise_dists,
#     loss_fn=lool_fn,
# )

parameters to be optimized: ['length_scale', 'smoothness', 'noise']
bounds: [[1.e-02 2.e+00]
 [1.e-01 4.e+00]
 [1.e-06 1.e+00]]
initial x0: [0.24467638 3.15178485 0.57179913]
|   iter    |  target   | length... | smooth... |   noise   |
-------------------------------------------------------------
| 1         | -29606.96 | 0.2446763 | 3.1517848 | 0.5717991 |
| 2         | -20551.34 | 1.9204344 | 2.8280969 | 0.9998672 |
| 3         | -24437.19 | 0.4479339 | 1.5081197 | 0.7398412 |
| 4         | -123232.4 | 1.9929468 | 1.3337532 | 0.1365454 |
| 5         | -45994.92 | 0.7741202 | 1.3500252 | 0.3664153 |
| 6         | -34961.13 | 1.4222066 | 3.6105554 | 0.5341159 |
| 7         | -32022.77 | 0.5021145 | 2.7200455 | 0.5617295 |
| 8         | -23455.00 | 1.0896941 | 3.5844456 | 0.8427797 |
| 9         | -27501.48 | 0.6189650 | 2.5615621 | 0.6802391 |
| 10        | -21577.34 | 1.9411508 | 3.5849118 | 0.9424259 |
| 11        | -76177.38 | 1.2880287 | 2.4971257 | 0.2276840 |
| 12        | -2335

In [9]:
#also optimize the scale
aqi_muygps_optimized = aqi_muygps_optimized.optimize_scale(
    batch_pairwise_dists,
    batch_nn_ys
)

# Inference

In [10]:
test_count = aqi_x_test.shape[0]
#get the indices of all the test points
test_indices = np.arange(test_count)
#get the indices of the neighbors for each of those test points as well
test_nn_indices, _ = nbrs_lookup.get_nns(aqi_x_test)

In [11]:
#create tesors to store the crosswise distances and pairwise distances between points and the point's neighbors.
#as well as the actual values for each points neighbors.
(
    test_crosswise_dists,
    test_pairwise_dists,
    test_nn_ys,
) = aqi_muygps.make_predict_tensors(
    test_indices,
    test_nn_indices,
    aqi_x_test,
    aqi_x_train,
    aqi_y_train,
)

In [12]:
#calculate kernels to store the crosswise covariance and pairwise covariance. these are used to know how much a points neighbors influence it.
kcross = aqi_muygps_optimized.kernel(test_crosswise_dists)
kin = aqi_muygps_optimized.kernel(test_pairwise_dists)

In [13]:
#Take the crosswise covariance and pairwise covariance to know how the neighbors effect each point,
#and then use the values of the points to calculate the actual value based on the neighbors influence.
predictions = aqi_muygps_optimized.posterior_mean(kin, kcross, test_nn_ys)
#because mean was removed from the datapoints before, need to add it back in.
predictions = predictions + aqi_y_mean
#calculate the variances, how much the guess is spread, meaning uncertainty.
variances = aqi_muygps_optimized.posterior_variance(kin, kcross)
#get the range that values need to be in in order to fall within 95% certainty.
confidence_intervals = np.sqrt(variances) * 1.96
#coverage is the proportion of guesses that differ from the true response by no more than the confidence interval size.
coverage = np.count_nonzero(np.abs(aqi_y_test - predictions) < confidence_intervals) / test_count

In [14]:
#print the results of the model
print_results(
    aqi_y_test, ("optimized", aqi_muygps_optimized, predictions, variances, confidence_intervals, coverage)
)

name,smoothness,length scale,noise variance,variance scale,rmse,mean variance,mean confidence interval,coverage
optimized,0.100000,0.010000,0.000001,224.722651,16.521558,192.224908,27.149088,0.890909
